In [1]:
import os
import re
import json
import time
import random
import faiss
import requests
import numpy as np

from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse, unquote
from getpass import getpass


# Configuração

MODEL = "google/embeddinggemma-300m"
DEEPINFRA_EMBEDDINGS_URL = "https://api.deepinfra.com/v1/openai/embeddings"

STORE_DIR = Path("wiki_faiss_store")
FAISS_PATH = STORE_DIR / "index.faiss"
DOCS_PATH = STORE_DIR / "docs.jsonl"
EMBEDDINGS_PATH = STORE_DIR / "embeddings.npy"
MANIFEST_PATH = STORE_DIR / "manifest.json"

USER_AGENT = "wiki-faiss-rag-notebook/0.1"


if not os.getenv("DEEPINFRA_TOKEN"):
    os.environ["DEEPINFRA_TOKEN"] = getpass("Cole seu DEEPINFRA_TOKEN: ")

In [2]:
# =========================
# Funções básicas
# =========================

def l2_normalize(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """
    Normaliza os embeddings para norma L2 = 1.

    Com isso:
      produto interno no FAISS = similaridade cosseno.
    """
    x = np.asarray(x, dtype=np.float32)

    if x.ndim == 1:
        x = x.reshape(1, -1)

    norms = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(norms, eps)


def page_key(lang: str, title: str) -> str:
    """
    Cria uma chave estável para uma página da Wikipédia.
    """
    clean_title = re.sub(r"\s+", "_", title.strip())
    return f"{lang}:{clean_title}"


def parse_wikipedia_page(page: str, default_lang: str = "pt"):
    """
    Aceita:
      - URL: https://pt.wikipedia.org/wiki/Albert_Einstein
      - título: Albert Einstein
      - page_key: pt:Albert_Einstein

    Retorna:
      lang, title
    """
    page = page.strip()

    # Caso já seja uma page_key
    if re.match(r"^[a-z-]+:.+", page) and not page.startswith("http"):
        lang, title = page.split(":", 1)
        return lang, title.replace("_", " ")

    # Caso seja URL
    if page.startswith("http://") or page.startswith("https://"):
        parsed = urlparse(page)
        host_parts = parsed.netloc.split(".")
        lang = host_parts[0] if len(host_parts) >= 3 else default_lang

        if "/wiki/" not in parsed.path:
            raise ValueError(f"URL não parece ser uma página /wiki/: {page}")

        title = parsed.path.split("/wiki/", 1)[1]
        title = unquote(title).replace("_", " ")
        return lang, title

    # Caso seja apenas título
    return default_lang, page


def request_with_retry(
    method: str,
    url: str,
    *,
    session: requests.Session | None = None,
    retry_statuses: tuple[int, ...] = (429, 500, 502, 503, 504),
    max_retries: int = 6,
    timeout_s: float = 30.0,
    backoff_initial_s: float = 1.0,
    backoff_max_s: float = 30.0,
    **kwargs,
) -> requests.Response:
    """
    Faz requisições HTTP com retry/backoff para evitar estourar rate limits.

    - Respeita header Retry-After quando presente.
    - Usa jitter para evitar thundering herd.
    """
    if max_retries < 1:
        raise ValueError("max_retries deve ser >= 1")

    sess = session or requests.Session()
    delay = float(backoff_initial_s)
    last_exc: Exception | None = None

    for attempt in range(1, max_retries + 1):
        try:
            r = sess.request(method, url, timeout=timeout_s, **kwargs)

            if r.status_code in retry_statuses:
                retry_after = r.headers.get("Retry-After")
                if retry_after:
                    try:
                        wait_s = float(retry_after)
                    except ValueError:
                        wait_s = delay
                else:
                    wait_s = delay

                wait_s = min(float(wait_s), float(backoff_max_s))
                if attempt >= max_retries:
                    r.raise_for_status()
                    return r

                # jitter proporcional, mas pequeno
                jitter = random.uniform(0.0, 0.25 * max(wait_s, 0.01))
                print(
                    f"[WARN] HTTP {r.status_code} ao acessar {url}. "
                    f"Tentando novamente em {wait_s + jitter:.1f}s "
                    f"(tentativa {attempt}/{max_retries})"
                )
                time.sleep(wait_s + jitter)
                delay = min(delay * 2.0, float(backoff_max_s))
                continue

            r.raise_for_status()
            return r

        except requests.RequestException as e:
            last_exc = e
            if attempt >= max_retries:
                raise

            wait_s = min(delay, float(backoff_max_s))
            jitter = random.uniform(0.0, 0.25 * max(wait_s, 0.01))
            print(
                f"[WARN] Erro de rede ({type(e).__name__}): {e}. "
                f"Retry em {wait_s + jitter:.1f}s (tentativa {attempt}/{max_retries})"
            )
            time.sleep(wait_s + jitter)
            delay = min(delay * 2.0, float(backoff_max_s))

    if last_exc:
        raise last_exc
    raise RuntimeError("Falha inesperada em request_with_retry")


def fetch_wikipedia_text(
    page: str,
    default_lang: str = "pt",
    *,
    request_timeout_s: float = 30.0,
    max_retries: int = 6,
    session: requests.Session | None = None,
) -> dict:
    """
    Baixa o texto limpo de uma página da Wikipédia via API MediaWiki.

    Retorna:
      {
        lang,
        title,
        page_key,
        url,
        text
      }
    """
    lang, title = parse_wikipedia_page(page, default_lang=default_lang)

    api_url = f"https://{lang}.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "explaintext": "1",
        "redirects": "1",
        "titles": title,
    }

    headers = {"User-Agent": USER_AGENT}

    r = request_with_retry(
        "GET",
        api_url,
        params=params,
        headers=headers,
        max_retries=max_retries,
        timeout_s=request_timeout_s,
        session=session,
    )
    data = r.json()

    pages = data["query"]["pages"]
    page_data = next(iter(pages.values()))

    if "missing" in page_data:
        raise ValueError(f"Página não encontrada: {page}")

    final_title = page_data["title"]
    text = page_data.get("extract", "").strip()

    if not text:
        raise ValueError(f"Página sem texto extraído: {page}")

    key = page_key(lang, final_title)
    url = f"https://{lang}.wikipedia.org/wiki/{final_title.replace(' ', '_')}"

    return {
        "lang": lang,
        "title": final_title,
        "page_key": key,
        "url": url,
        "text": text,
    }


def chunk_text(text: str, chunk_words: int = 180, overlap_words: int = 40) -> list[str]:
    """
    Divide texto em chunks simples por número de palavras.

    É uma estratégia simples e auditável:
      - chunk_words define o tamanho aproximado.
      - overlap_words preserva contexto entre chunks consecutivos.
    """
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    words = text.split()

    if not words:
        return []

    if chunk_words <= overlap_words:
        raise ValueError("chunk_words deve ser maior que overlap_words.")

    chunks = []
    step = chunk_words - overlap_words

    for start in range(0, len(words), step):
        end = start + chunk_words
        chunk = " ".join(words[start:end]).strip()

        # Evita chunk final minúsculo, exceto se for o único.
        if len(chunk.split()) >= 30 or not chunks:
            chunks.append(chunk)

        if end >= len(words):
            break

    return chunks


def deepinfra_embed(texts: list[str], batch_size: int = 16) -> np.ndarray:
    """
    Gera embeddings via DeepInfra.

    Retorna:
      matriz float32 de shape [n_textos, dim]
    """
    token = os.getenv("DEEPINFRA_TOKEN")

    if not token:
        raise RuntimeError("Defina DEEPINFRA_TOKEN no ambiente.")

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        payload = {
            "model": MODEL,
            "input": batch,
            "encoding_format": "float",
        }

        r = requests.post(
            DEEPINFRA_EMBEDDINGS_URL,
            headers=headers,
            json=payload,
            timeout=60,
        )
        r.raise_for_status()

        data = r.json()
        batch_embeddings = [item["embedding"] for item in data["data"]]
        all_embeddings.extend(batch_embeddings)

    return np.array(all_embeddings, dtype=np.float32)

In [3]:
def load_docs() -> list[dict]:
    """
    Carrega docs.jsonl.
    Cada linha é um chunk com metadados.
    """
    if not DOCS_PATH.exists():
        return []

    docs = []
    with open(DOCS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                docs.append(json.loads(line))

    return docs


def save_docs(docs: list[dict]):
    STORE_DIR.mkdir(parents=True, exist_ok=True)

    with open(DOCS_PATH, "w", encoding="utf-8") as f:
        for doc in docs:
            f.write(json.dumps(doc, ensure_ascii=False) + "\n")


def load_embeddings() -> np.ndarray:
    """
    Carrega embeddings.npy.
    """
    if not EMBEDDINGS_PATH.exists():
        return np.zeros((0, 0), dtype=np.float32)

    return np.load(EMBEDDINGS_PATH).astype(np.float32)


def save_embeddings(embeddings: np.ndarray):
    STORE_DIR.mkdir(parents=True, exist_ok=True)
    np.save(EMBEDDINGS_PATH, embeddings.astype(np.float32))


def build_faiss_index(embeddings: np.ndarray):
    """
    Cria um índice FAISS IndexFlatIP.

    Como os embeddings estão normalizados:
      IndexFlatIP = cosine similarity.
    """
    if embeddings.size == 0 or embeddings.shape[0] == 0:
        return None

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings.astype(np.float32))
    return index


def load_faiss_index(embeddings: np.ndarray):
    """
    Carrega o índice FAISS se existir.
    Caso contrário, reconstrói a partir dos embeddings salvos.
    """
    if FAISS_PATH.exists():
        return faiss.read_index(str(FAISS_PATH))

    return build_faiss_index(embeddings)


def save_faiss_index(index):
    STORE_DIR.mkdir(parents=True, exist_ok=True)

    if index is not None:
        faiss.write_index(index, str(FAISS_PATH))


def save_manifest(extra: dict | None = None):
    STORE_DIR.mkdir(parents=True, exist_ok=True)

    manifest = {
        "model": MODEL,
        "faiss_index_type": "IndexFlatIP",
        "similarity": "cosine_similarity_via_l2_normalized_inner_product",
        "updated_at": datetime.now().isoformat(timespec="seconds"),
    }

    if extra:
        manifest.update(extra)

    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)


def load_store() -> dict:
    """
    Carrega todos os arquivos do índice local.
    """
    docs = load_docs()
    embeddings = load_embeddings()

    if len(docs) != embeddings.shape[0]:
        raise RuntimeError(
            f"Inconsistência no índice: {len(docs)} docs, "
            f"mas {embeddings.shape[0]} embeddings."
        )

    index = load_faiss_index(embeddings)

    return {
        "docs": docs,
        "embeddings": embeddings,
        "index": index,
    }


def save_store(docs: list[dict], embeddings: np.ndarray, index):
    """
    Salva metadados, embeddings e índice FAISS.
    """
    save_docs(docs)
    save_embeddings(embeddings)
    save_faiss_index(index)

    save_manifest({
        "num_docs": len(docs),
        "embedding_dim": int(embeddings.shape[1]) if embeddings.size else None,
    })


def remove_page_from_store(
    docs: list[dict],
    embeddings: np.ndarray,
    key: str,
):
    """
    Remove todos os chunks de uma página.

    Como IndexFlatIP não é pensado para remoção arbitrária simples,
    removemos dos arrays e depois reconstruímos o FAISS.
    """
    keep_indices = [i for i, doc in enumerate(docs) if doc["page_key"] != key]

    new_docs = [docs[i] for i in keep_indices]

    if len(keep_indices) == 0:
        new_embeddings = np.zeros((0, 0), dtype=np.float32)
    else:
        new_embeddings = embeddings[keep_indices]

    new_index = build_faiss_index(new_embeddings)

    return new_docs, new_embeddings, new_index

In [4]:
def index_wikipedia_pages(
    pages: list[str],
    default_lang: str = "pt",
    chunk_words: int = 180,
    overlap_words: int = 40,
    batch_size: int = 16,
    replace_existing_pages: bool = False,
    sleep_between_pages_s: float = 0.8,
    wikipedia_request_timeout_s: float = 30.0,
    wikipedia_max_retries: int = 6,
    continue_on_error: bool = True,
 ):
    """
    Indexa várias páginas da Wikipédia usando FAISS, com checkpoint por página.

    Objetivo:
      - Se der erro no meio, o que já foi indexado fica salvo em disco.
      - Ao rodar de novo, ele pula o que já foi indexado e evita chamar a API.

    Anti-rate-limit:
      - sleep_between_pages_s adiciona intervalo entre páginas.
      - wikipedia_max_retries tenta novamente em caso de 429/5xx com backoff.
    """
    def _norm_title(s: str) -> str:
        return re.sub(r"\s+", " ", s).strip().lower()

    store = load_store()
    docs = store["docs"]
    embeddings = store["embeddings"]
    index = store["index"]

    existing_page_keys = set(doc["page_key"] for doc in docs)
    existing_source_pages = set(
        doc.get("source_page")
        for doc in docs
        if isinstance(doc, dict) and doc.get("source_page")
    )
    existing_titles = {
        (doc.get("lang"), _norm_title(doc.get("title", ""))): doc.get("page_key")
        for doc in docs
        if isinstance(doc, dict) and doc.get("lang") and doc.get("title") and doc.get("page_key")
    }

    wiki_session = requests.Session()
    errors: list[dict] = []

    def _checkpoint_save():
        save_store(docs, embeddings, index)

    for page_i, page in enumerate(pages, start=1):
        # Se já indexamos esse input exatamente, não precisa nem chamar a API.
        if not replace_existing_pages and page in existing_source_pages:
            print(f"[SKIP] Já indexada (source_page): {page}")
            continue

        # Tentativa barata de detectar já-indexado sem bater na API.
        if not replace_existing_pages:
            try:
                # page_key exata
                if re.match(r"^[a-z-]+:.+", page) and not page.startswith("http") and page in existing_page_keys:
                    print(f"[SKIP] Já indexada (page_key): {page}")
                    continue

                lang_guess, title_guess = parse_wikipedia_page(page, default_lang=default_lang)
                candidate_key = page_key(lang_guess, title_guess)
                if candidate_key in existing_page_keys:
                    print(f"[SKIP] Já indexada (candidate_key): {candidate_key}")
                    continue

                if (lang_guess, _norm_title(title_guess)) in existing_titles:
                    key0 = existing_titles[(lang_guess, _norm_title(title_guess))]
                    print(f"[SKIP] Já indexada (título): {key0}")
                    continue
            except Exception:
                # Se algo inesperado acontecer nessa checagem, só segue o fluxo normal.
                pass

        if sleep_between_pages_s and page_i > 1:
            time.sleep(float(sleep_between_pages_s))

        try:
            page_obj = fetch_wikipedia_text(
                page,
                default_lang=default_lang,
                request_timeout_s=wikipedia_request_timeout_s,
                max_retries=wikipedia_max_retries,
                session=wiki_session,
            )
        except Exception as e:
            msg = f"{type(e).__name__}: {e}"
            errors.append({"page": page, "stage": "fetch", "error": msg})
            print(f"[ERROR] Falha ao baixar '{page}': {msg}")
            if continue_on_error:
                continue
            raise

        key = page_obj["page_key"]

        # Se a página já está indexada (pelo key final da API), decide o que fazer.
        if key in existing_page_keys and not replace_existing_pages:
            print(f"[SKIP] Já indexada: {key}")
            existing_source_pages.add(page)
            continue

        if key in existing_page_keys and replace_existing_pages:
            print(f"[REINDEX] Removendo versão anterior: {key}")
            docs, embeddings, index = remove_page_from_store(docs, embeddings, key)
            existing_page_keys = set(doc["page_key"] for doc in docs)
            existing_source_pages = set(
                doc.get("source_page")
                for doc in docs
                if isinstance(doc, dict) and doc.get("source_page")
            )
            existing_titles = {
                (doc.get("lang"), _norm_title(doc.get("title", ""))): doc.get("page_key")
                for doc in docs
                if isinstance(doc, dict) and doc.get("lang") and doc.get("title") and doc.get("page_key")
            }

        chunks = chunk_text(
            page_obj["text"],
            chunk_words=chunk_words,
            overlap_words=overlap_words,
        )

        if not chunks:
            print(f"[WARN] Sem chunks para: {key}")
            existing_page_keys.add(key)
            existing_source_pages.add(page)
            existing_titles[(page_obj["lang"], _norm_title(page_obj["title"]))] = key
            _checkpoint_save()
            continue

        new_docs = []
        for chunk_id, chunk in enumerate(chunks):
            new_docs.append({
                "doc_id": f"{key}#chunk={chunk_id}",
                "page_key": key,
                "lang": page_obj["lang"],
                "title": page_obj["title"],
                "url": page_obj["url"],
                "chunk_id": chunk_id,
                "source_page": page,
                "text": chunk,
            })

        try:
            new_texts = [doc["text"] for doc in new_docs]
            new_embeddings = deepinfra_embed(new_texts, batch_size=min(batch_size, len(new_texts)))
            new_embeddings = l2_normalize(new_embeddings)
        except Exception as e:
            msg = f"{type(e).__name__}: {e}"
            errors.append({"page": page, "stage": "embed", "error": msg})
            print(f"[ERROR] Falha ao gerar embeddings para '{key}': {msg}")
            if continue_on_error:
                continue
            raise

        if embeddings.size == 0 or embeddings.shape[0] == 0:
            embeddings = new_embeddings
            index = build_faiss_index(embeddings)
        else:
            embeddings = np.vstack([embeddings, new_embeddings])
            if index is None:
                index = build_faiss_index(embeddings)
            else:
                index.add(new_embeddings.astype(np.float32))

        docs.extend(new_docs)
        existing_page_keys.add(key)
        existing_source_pages.add(page)
        existing_titles[(page_obj["lang"], _norm_title(page_obj["title"]))] = key

        _checkpoint_save()
        print(f"[OK] {key}: {len(chunks)} chunks | checkpoint salvo")

    if errors:
        print("\nResumo de erros:")
        for err in errors:
            print(f"  - {err['stage']} | {err['page']} | {err['error']}")

    return {
        "docs": docs,
        "embeddings": embeddings,
        "index": index,
        "errors": errors,
    }

In [5]:
import pandas as pd

pages = pd.read_json("pairs.json")["wiki"].tolist()

In [6]:
# pages = [
#     "https://pt.wikipedia.org/wiki/Universidade_Estadual_de_Campinas"
# ]

store = index_wikipedia_pages(
    pages,
    chunk_words=180,
    overlap_words=40,
    batch_size=16,
    replace_existing_pages=False,
)

[SKIP] Já indexada (source_page): https://pt.wikipedia.org/wiki/Bolsa_Fam%C3%ADlia
[SKIP] Já indexada (source_page): https://pt.wikipedia.org/wiki/Direitos_LGBT_no_Brasil
[SKIP] Já indexada (source_page): https://pt.wikipedia.org/wiki/Reforma_agr%C3%A1ria_no_Brasil
[SKIP] Já indexada (source_page): https://pt.wikipedia.org/wiki/Habita%C3%A7%C3%A3o_social
[SKIP] Já indexada (source_page): https://pt.wikipedia.org/wiki/Intervencionismo_econ%C3%B4mico
[SKIP] Já indexada (source_page): https://pt.wikipedia.org/wiki/Privatiza%C3%A7%C3%A3o
[SKIP] Já indexada (source_page): https://pt.wikipedia.org/wiki/Sal%C3%A1rio_m%C3%ADnimo
[SKIP] Já indexada (source_page): https://pt.wikipedia.org/wiki/D%C3%ADvida_p%C3%BAblica
[SKIP] Já indexada (source_page): https://pt.wikipedia.org/wiki/Controle_de_armas
[SKIP] Já indexada (source_page): https://pt.wikipedia.org/wiki/Maioridade_penal
[SKIP] Já indexada (source_page): https://pt.wikipedia.org/wiki/Mil%C3%ADcia_(crime)
[SKIP] Já indexada (source_page): 

In [7]:
def list_indexed_pages():
    store = load_store()
    docs = store["docs"]

    pages = {}

    for doc in docs:
        key = doc["page_key"]

        if key not in pages:
            pages[key] = {
                "title": doc["title"],
                "lang": doc["lang"],
                "url": doc["url"],
                "chunks": 0,
            }

        pages[key]["chunks"] += 1

    for key, info in sorted(pages.items()):
        print(key)
        print(f"  título: {info['title']}")
        print(f"  chunks: {info['chunks']}")
        print(f"  url: {info['url']}")
        print()

    return pages


indexed_pages = list_indexed_pages()

pt:Agronegócio_no_Brasil
  título: Agronegócio no Brasil
  chunks: 18
  url: https://pt.wikipedia.org/wiki/Agronegócio_no_Brasil

pt:Aquecimento_global
  título: Aquecimento global
  chunks: 184
  url: https://pt.wikipedia.org/wiki/Aquecimento_global

pt:Autoridade
  título: Autoridade
  chunks: 6
  url: https://pt.wikipedia.org/wiki/Autoridade

pt:Ação_afirmativa
  título: Ação afirmativa
  chunks: 16
  url: https://pt.wikipedia.org/wiki/Ação_afirmativa

pt:Bolsa_Família
  título: Bolsa Família
  chunks: 35
  url: https://pt.wikipedia.org/wiki/Bolsa_Família

pt:Controle_de_armamento
  título: Controle de armamento
  chunks: 3
  url: https://pt.wikipedia.org/wiki/Controle_de_armamento

pt:Corrupção_no_Brasil
  título: Corrupção no Brasil
  chunks: 84
  url: https://pt.wikipedia.org/wiki/Corrupção_no_Brasil

pt:Criminalidade_no_Brasil
  título: Criminalidade no Brasil
  chunks: 48
  url: https://pt.wikipedia.org/wiki/Criminalidade_no_Brasil

pt:Democracia_participativa
  título: Democra

In [8]:
def resolve_page_key_for_retrieval(
    page: str,
    docs: list[dict],
    default_lang: str = "pt",
) -> str:
    """
    Resolve o identificador da página.

    Aceita:
      - page_key exata: pt:Exame_Nacional_do_Ensino_Médio
      - URL
      - título aproximado
    """
    available_keys = sorted(set(doc["page_key"] for doc in docs))

    # Caso 1: page_key exata
    if page in available_keys:
        return page

    # Caso 2: URL ou título
    lang, title = parse_wikipedia_page(page, default_lang=default_lang)
    candidate_key = page_key(lang, title)

    if candidate_key in available_keys:
        return candidate_key

    # Caso 3: comparação por título normalizado
    wanted = title.lower().replace("_", " ").strip()

    matches = []
    for key in available_keys:
        doc = next(d for d in docs if d["page_key"] == key)
        doc_title = doc["title"].lower().replace("_", " ").strip()

        if wanted == doc_title or wanted in doc_title or doc_title in wanted:
            matches.append(key)

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        raise ValueError(
            "Mais de uma página compatível encontrada. Use uma page_key exata:\n"
            + "\n".join(matches)
        )

    raise ValueError(
        "Página não encontrada no índice.\n\n"
        "Páginas disponíveis:\n"
        + "\n".join(available_keys)
    )

In [9]:
def resolve_page_key_for_retrieval(
    page: str,
    docs: list[dict],
    default_lang: str = "pt",
) -> str:
    """
    Resolve o identificador da página.

    Aceita:
      - page_key exata: pt:Exame_Nacional_do_Ensino_Médio
      - URL
      - título aproximado
    """
    available_keys = sorted(set(doc["page_key"] for doc in docs))

    # Caso 1: page_key exata
    if page in available_keys:
        return page

    # Caso 2: URL ou título
    lang, title = parse_wikipedia_page(page, default_lang=default_lang)
    candidate_key = page_key(lang, title)

    if candidate_key in available_keys:
        return candidate_key

    # Caso 3: comparação por título normalizado
    wanted = title.lower().replace("_", " ").strip()

    matches = []
    for key in available_keys:
        doc = next(d for d in docs if d["page_key"] == key)
        doc_title = doc["title"].lower().replace("_", " ").strip()

        if wanted == doc_title or wanted in doc_title or doc_title in wanted:
            matches.append(key)

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        raise ValueError(
            "Mais de uma página compatível encontrada. Use uma page_key exata:\n"
            + "\n".join(matches)
        )

    raise ValueError(
        "Página não encontrada no índice.\n\n"
        "Páginas disponíveis:\n"
        + "\n".join(available_keys)
    )


def retrieve_from_wikipedia_page(
    prompt: str,
    page: str,
    top_n: int = 5,
    default_lang: str = "pt",
) -> list[dict]:
    """
    Recupera os top-N chunks mais relevantes dentro de uma página específica.

    Esta função:
      1. carrega o índice persistido;
      2. localiza os chunks da página escolhida;
      3. gera embedding do prompt;
      4. cria um índice FAISS temporário só com os chunks daquela página;
      5. retorna os top-N chunks por similaridade cosseno.
    """
    store = load_store()

    docs = store["docs"]
    embeddings = store["embeddings"]

    if len(docs) == 0:
        raise RuntimeError("O índice está vazio. Rode index_wikipedia_pages primeiro.")

    key = resolve_page_key_for_retrieval(
        page=page,
        docs=docs,
        default_lang=default_lang,
    )

    page_indices = [
        i for i, doc in enumerate(docs)
        if doc["page_key"] == key
    ]

    if not page_indices:
        raise ValueError(f"Nenhum chunk encontrado para a página: {key}")

    query_embedding = deepinfra_embed([prompt], batch_size=1)
    query_embedding = l2_normalize(query_embedding).astype(np.float32)

    page_embeddings = embeddings[page_indices].astype(np.float32)

    dim = page_embeddings.shape[1]
    page_index = faiss.IndexFlatIP(dim)
    page_index.add(page_embeddings)

    k = min(top_n, len(page_indices))
    scores, local_positions = page_index.search(query_embedding, k)

    results = []

    for rank, local_pos in enumerate(local_positions[0], start=1):
        if local_pos < 0:
            continue

        global_idx = page_indices[local_pos]
        doc = docs[global_idx]

        results.append({
            "rank": rank,
            "score": float(scores[0][rank - 1]),
            "doc_id": doc["doc_id"],
            "page_key": doc["page_key"],
            "title": doc["title"],
            "url": doc["url"],
            "chunk_id": doc["chunk_id"],
            "text": doc["text"],
        })

    return results


def print_retrieval_results(results: list[dict], max_chars: int = 1200):
    for result in results:
        print("=" * 90)
        print(f"Rank: {result['rank']} | Score: {result['score']:.4f}")
        print(f"Página: {result['title']}")
        print(f"Page key: {result['page_key']}")
        print(f"Chunk: {result['chunk_id']}")
        print(f"URL: {result['url']}")
        print("-" * 90)

        text = result["text"]

        if len(text) > max_chars:
            print(text[:max_chars] + "...")
        else:
            print(text)

        print()

In [10]:
results = retrieve_from_wikipedia_page(
    prompt="Quando a Unicamp foi fundada e onde fica localizada?",
    page="Universidade Estadual de Campinas",
    top_n=2,
)

print_retrieval_results(results)

Rank: 1 | Score: 0.6489
Página: Universidade Estadual de Campinas
Page key: pt:Universidade_Estadual_de_Campinas
Chunk: 0
URL: https://pt.wikipedia.org/wiki/Universidade_Estadual_de_Campinas
------------------------------------------------------------------------------------------
Universidade Estadual de Campinas (Unicamp) é uma instituição de ensino superior pública estadual brasileira, sediada na cidade de Campinas, no estado de São Paulo, considerada uma das melhores universidades do país e da América Latina. É uma das quatro universidades mantidas pelo Governo do Estado de São Paulo, ao lado da Universidade de São Paulo (USP), da Universidade Estadual Paulista (Unesp) e da Universidade Virtual do Estado de São Paulo (Univesp). Fundada em 1962, a Unicamp foi projetada do zero como um sistema integrado de centros de pesquisa, ao contrário de outras universidades brasileiras, geralmente criadas pela consolidação das escolas e institutos anteriormente existentes. Seu foco em pesquisa 

In [ ]:
def build_context(results: list[dict]) -> str:
    blocks = []

    for r in results:
        block = (
            f"- {r['text']}"
        )
        blocks.append(block)

    return "Página: {r['title']}\n URL: {r['url']}\n \n\n".join(blocks)


context = build_context(results)

print(context[:4000])



--------------------------------------------------------------------------------[Documento 1]
Página: Universidade Estadual de Campinas
URL: https://pt.wikipedia.org/wiki/Universidade_Estadual_de_Campinas
Chunk: 0
Score: 0.6489

Universidade Estadual de Campinas (Unicamp) é uma instituição de ensino superior pública estadual brasileira, sediada na cidade de Campinas, no estado de São Paulo, considerada uma das melhores universidades do país e da América Latina. É uma das quatro universidades mantidas pelo Governo do Estado de São Paulo, ao lado da Universidade de São Paulo (USP), da Universidade Estadual Paulista (Unesp) e da Universidade Virtual do Estado de São Paulo (Univesp). Fundada em 1962, a Unicamp foi projetada do zero como um sistema integrado de centros de pesquisa, ao contrário de outras universidades brasileiras, geralmente criadas pela consolidação das escolas e institutos anteriormente existentes. Seu foco em pesquisa reflete que quase metade de seus estudantes são alu